# <u>***Иследовательский анализ лояльности пользователей Яндекс Афиши***</u>

- Автор: Якунин Михаил Евгеньевич
- Дата: __ марта 2026 г.

## *Цели и задачи проекта*

<u>**Цель исследования:**</u>


<u>**Задачи проекта:**</u>

## *Описание данных*

Выгрузка из базы данных `SQL` следующие данные:
- `user_id` — уникальный идентификатор пользователя, совершившего заказ;
- `device_type_canonical` — тип устройства, с которого был оформлен заказ ( `mobile` — мобильные устройства, `desktop` — стационарные);
- `order_id` — уникальный идентификатор заказа;
- `order_dt` — дата создания заказа (используйте данные `created_dt_msk` );
- `order_ts` — дата и время создания заказа (используйте данные `created_ts_msk` );
- `currency_code` — валюта оплаты;
- `revenue` — выручка от заказа;
- `tickets_count` — количество купленных билетов;
- `days_since_prev` — количество дней от предыдущей покупки пользователя, для пользователей с одной покупкой — значение пропущено;
- `event_id` — уникальный идентификатор мероприятия;
- `service_name` — название билетного оператора;
- `event_type_main` — основной тип мероприятия (театральная постановка, концерт и так далее);
- `region_name` — название региона, в котором прошло мероприятие;
- `city_name` — название города, в котором прошло мероприятие.

## *Содержание*

1. [Загрузка данных и знакомство с ними](#1)
2. [Предобработка данных](#2)
3. [Исследовательский анализ данных](#3)
4. [Итоговый вывод и рекомендации](#4)

## 1. Загрузка данных и знакомство с ними <a id=1></a>

### 1.1 *Загрузка данных*

In [1]:
import pandas as pd
from sqlalchemy import create_engine
import matplotlib.pyplot as plt
import seaborn as sns
from phik import phik_matrix 

In [2]:
db_config = {'user': 'praktikum_student', # имя пользователя
             'pwd': 'Sdf4$2;d-d30pp', # пароль
             'host': 'rc1b-wcoijxj3yxfsf3fs.mdb.yandexcloud.net',
             'port': 6432, # порт подключения
             'db': 'data-analyst-afisha' # название базы данных
             }

In [3]:
connection_string = 'postgresql://{}:{}@{}:{}/{}'.format(
    db_config['user'],
    db_config['pwd'],
    db_config['host'],
    db_config['port'],
    db_config['db'],
) 

In [4]:
engine = create_engine(connection_string) 

In [5]:
query = '''
SELECT
    user_id,
    device_type_canonical,
    order_id,
    created_dt_msk as order_dt,
    created_ts_msk as order_ts,
    currency_code,
    revenue,
    tickets_count,
    extract(day from created_dt_msk - LAG(created_dt_msk) OVER (PARTITION BY user_id ORDER BY created_dt_msk)) AS days_since_prev,
    event_id,
    event_name_code as event_name,
    event_type_main,
    service_name,
    region_name,
    city_name
FROM afisha.purchases as p
INNER JOIN afisha.events as e using(event_id)
INNER JOIN afisha.city as city using(city_id)
INNER JOIN afisha.regions as r using(region_id)
WHERE device_type_canonical in ('mobile', 'desktop') AND event_type_main != 'фильм'
ORDER BY user_id
''' 

In [6]:
df = pd.read_sql_query(query, con=engine) 

### 1.2 *Знакомство с загруженными даннами*

In [7]:
df.head(5)

,user_id,device_type_canonical,order_id,order_dt,order_ts,currency_code,revenue,tickets_count,days_since_prev,event_id,event_name,event_type_main,service_name,region_name,city_name
0,0002849b70a3ce2,mobile,4359165,2024-08-20,2024-08-20 16:08:03,rub,1521.94,4,NaN,169230,f0f7b271-04eb-4af6-bcb8-8f05cf46d6ad,театр,Край билетов,Каменевский регион,Глиногорск
1,0005ca5e93f2cf4,mobile,7965605,2024-07-23,2024-07-23 18:36:24,rub,289.45,2,NaN,237325,40efeb04-81b7-4135-b41f-708ff00cc64c,выставки,Мой билет,Каменевский регион,Глиногорск
2,0005ca5e93f2cf4,mobile,7292370,2024-10-06,2024-10-06 13:56:02,rub,1258.57,4,75.0,578454,01f3fb7b-ed07-4f94-b1d3-9a2e1ee5a8ca,другое,За билетом!,Каменевский регион,Глиногорск
3,000898990054619,mobile,1139875,2024-07-13,2024-07-13 19:40:48,rub,8.49,2,NaN,387271,2f638715-8844-466c-b43f-378a627c419f,другое,Лови билет!,Североярская область,Озёрск
4,000898990054619,mobile,972400,2024-10-04,2024-10-04 22:33:15,rub,1390.41,3,83.0,509453,10d805d3-9809-4d8a-834e-225b7d03f95d,стендап,Билеты без проблем,Озернинский край,Родниковецк


In [8]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 290611 entries, 0 to 290610
Data columns (total 15 columns):
 #   Column                 Non-Null Count   Dtype         
---  ------                 --------------   -----         
 0   user_id                290611 non-null  str           
 1   device_type_canonical  290611 non-null  str           
 2   order_id               290611 non-null  int64         
 3   order_dt               290611 non-null  datetime64[us]
 4   order_ts               290611 non-null  datetime64[us]
 5   currency_code          290611 non-null  str           
 6   revenue                290611 non-null  float64       
 7   tickets_count          290611 non-null  int64         
 8   days_since_prev        268678 non-null  float64       
 9   event_id               290611 non-null  int64         
 10  event_name             290611 non-null  str           
 11  event_type_main        290611 non-null  str           
 12  service_name           290611 non-null  str           


In [9]:
df.describe(include='all')

,user_id,device_type_canonical,order_id,order_dt,order_ts,currency_code,revenue,tickets_count,days_since_prev,event_id,event_name,event_type_main,service_name,region_name,city_name
count,290611,290611,2.906110e+05,290611,290611,290611,290611.000000,290611.000000,268678.000000,290611.000000,290611,290611,290611,290611,290611
unique,21933,2,NaN,NaN,NaN,2,NaN,NaN,NaN,NaN,15248,7,36,81,352
top,0beb8fc0c0a9ce1,mobile,NaN,NaN,NaN,rub,NaN,NaN,NaN,NaN,9cc55c15-4375-4129-9979-3129688ba1b4,концерты,Билеты без проблем,Каменевский регион,Глиногорск
freq,10251,232490,NaN,NaN,NaN,285542,NaN,NaN,NaN,NaN,3953,115276,63519,91058,89446
mean,NaN,NaN,4.326225e+06,2024-09-01 22:36:38.741272,2024-09-02 13:31:19.397731,NaN,625.584360,2.754311,3.222381,438019.834992,NaN,NaN,NaN,NaN,NaN
min,NaN,NaN,1.000000e+00,2024-06-01 00:00:00,2024-06-01 00:00:42,NaN,-90.760000,1.000000,0.000000,4436.000000,NaN,NaN,NaN,NaN,NaN
25%,NaN,NaN,2.163618e+06,2024-07-30 00:00:00,2024-07-30 11:53:37.500000,NaN,116.850000,2.000000,0.000000,361772.000000,NaN,NaN,NaN,NaN,NaN
50%,NaN,NaN,4.326366e+06,2024-09-12 00:00:00,2024-09-12 14:02:10,NaN,356.010000,3.000000,0.000000,498275.000000,NaN,NaN,NaN,NaN,NaN
75%,NaN,NaN,6.488330e+06,2024-10-09 00:00:00,2024-10-09 15:57:55.500000,NaN,810.130000,4.000000,1.000000,546287.000000,NaN,NaN,NaN,NaN,NaN
max,NaN,NaN,8.653108e+06,2024-10-31 00:00:00,2024-10-31 23:59:54,NaN,81174.540000,57.000000,148.000000,592325.000000,NaN,NaN,NaN,NaN,NaN


### 1.3 *Промежуточный вывод*

*Общая характеристика датасета*

* Объем данных: 290 611 записей о покупках билетов
* Временной период: данные с 1 июня по 31 октября 2024 года (5 месяцев)
* Пользовательская база: 21 933 уникальных пользователя
* География: 81 регион, 352 города

*Качество данных*
* Полнота данных: высокое
* Типы данных: корректно приведены, все поля заполнены
* Для оптимизации занемаемого `dataframe` объема памяти можно понизить разряд данных в столбцах `order_id`, `tickets_count`, `event_id`

*Аномалии для исследования*
* Отрицательные значения revenue (возвраты/ошибки?)
* Экстремально высокие чеки (>80k₽)
* Выбросы по количеству билетов (57 шт. в одном заказе)
* Длительные периоды между покупками (до 148 дней)

## 2. Предобработка данных <a id=2></a>

### 2.1 *Конвертация валют и создание признака revenue_rub*

In [10]:
exchange_rate_tenge = pd.read_csv('final_tickets_tenge_df.csv')

In [11]:
exchange_rate_tenge.head(5)

,data,nominal,curs,cdx
0,2024-01-10,100,19.9391,kzt
1,2024-01-11,100,19.7255,kzt
2,2024-01-12,100,19.5839,kzt
3,2024-01-13,100,19.4501,kzt
4,2024-01-14,100,19.4501,kzt


In [12]:
exchange_rate_tenge = exchange_rate_tenge.set_index('data')['curs']
exchange_rate_tenge.index = pd.to_datetime(exchange_rate_tenge.index)

In [24]:
def calculate_revenue_rub(row):
    if row['currency_code'] == 'kzt':
        exchange_rate = exchange_rate_tenge.get(row['order_dt'])
        return round(row['revenue'] / 100 * exchange_rate, 2)
    else:
        return round(row['revenue'], 2)

In [25]:
df['revenue_rub'] = df.apply(calculate_revenue_rub, axis=1)

In [27]:
df.sample(5)

,user_id,device_type_canonical,order_id,order_dt,order_ts,currency_code,revenue,tickets_count,days_since_prev,event_id,event_name,event_type_main,service_name,region_name,city_name,revenue_rub
288817,fe237d2cfd6e503,mobile,7058601,2024-10-11,2024-10-11 11:01:18,rub,1694.449951,4,0.0,560986,787d61f0-d0fa-4f32-97b4-e0079b51b6b0,концерты,Облачко,Светополянский округ,Глиноград,1694.45
251179,d9dfd8d6cc71fb4,mobile,2387049,2024-09-07,2024-09-07 09:43:13,rub,763.020020,3,50.0,309205,b4cd713e-2541-4589-a58a-8aaa4f502059,концерты,Билеты без проблем,Каменевский регион,Глиногорск,763.02
61899,2a92d67b4597d1c,mobile,6480138,2024-10-30,2024-10-30 08:24:59,rub,36.360001,3,0.0,583621,c4a070cd-3a10-4c65-99e3-1d7a03a18bb6,другое,Билеты без проблем,Солнечноземская область,Глинополье,36.36
290009,ff0d86940a3fe67,mobile,3464573,2024-09-25,2024-09-25 22:06:19,rub,7.520000,1,0.0,400644,a278afa0-578d-4772-b479-e42abfdbd8a3,театр,Билеты без проблем,Североключевской округ,Зоречанск,7.52
272801,efd7b595af23463,mobile,3166743,2024-10-09,2024-10-09 08:21:33,rub,782.030029,3,0.0,455446,d4578327-04a6-49e6-99fb-282651ef0c9e,концерты,Облачко,Лугоградская область,Кристалевск,782.03


### 2.2 *Обработка пропусков, типов данных и выбросов*

In [16]:
df.isna().sum().reset_index(name='Кол-во') \
    .rename(columns={'index': 'Столбец'}) \
    .assign(Процент=lambda x: x['Кол-во'] / len(df) * 100) \
    .sort_values(by='Процент', ascending=False) \
    .reset_index(drop=True) \
    .style.format({'Процент': '{:.2f}%'})

,Столбец,Кол-во,Процент
0,days_since_prev,21933,7.55%
1,user_id,0,0.00%
2,order_id,0,0.00%
3,device_type_canonical,0,0.00%
4,order_ts,0,0.00%
5,currency_code,0,0.00%
6,revenue,0,0.00%
7,order_dt,0,0.00%
8,tickets_count,0,0.00%
9,event_id,0,0.00%


In [17]:
df['days_since_prev'] = df['days_since_prev'].fillna(-1)

In [20]:
df['order_id'] = df['order_id'].astype('int32')
df['tickets_count'] = df['tickets_count'].astype('int8')
df['event_id'] = df['event_id'].astype('int32')
df['revenue'] = df['revenue'].astype('float32')
df['days_since_prev'] = df['days_since_prev'].astype('float32')
df['revenue_rub'] = df['revenue_rub'].astype('float32')

In [21]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 290611 entries, 0 to 290610
Data columns (total 16 columns):
 #   Column                 Non-Null Count   Dtype         
---  ------                 --------------   -----         
 0   user_id                290611 non-null  str           
 1   device_type_canonical  290611 non-null  str           
 2   order_id               290611 non-null  int32         
 3   order_dt               290611 non-null  datetime64[us]
 4   order_ts               290611 non-null  datetime64[us]
 5   currency_code          290611 non-null  str           
 6   revenue                290611 non-null  float32       
 7   tickets_count          290611 non-null  int8          
 8   days_since_prev        290611 non-null  float32       
 9   event_id               290611 non-null  int32         
 10  event_name             290611 non-null  str           
 11  event_type_main        290611 non-null  str           
 12  service_name           290611 non-null  str           


In [62]:
for column in df.columns:
    try:
        df[column] = df[column].str.lower()
    except:
        continue

In [67]:
max_col_len = max(len(col) for col in df.columns)
max_duplicates_len = max(len(str(df[col].duplicated().sum())) for col in df.columns)

for column in df.columns:
    duplicates = df[column].duplicated().sum()
    print(f'В столбце {column:<{max_col_len}} найдено  {duplicates:>{max_duplicates_len}}  дубликатов')

В столбце user_id               найдено  268678  дубликатов
В столбце device_type_canonical найдено  290609  дубликатов
В столбце order_id              найдено       0  дубликатов
В столбце order_dt              найдено  290458  дубликатов
В столбце order_ts              найдено    9858  дубликатов
В столбце currency_code         найдено  290609  дубликатов
В столбце revenue               найдено  248089  дубликатов
В столбце tickets_count         найдено  290589  дубликатов
В столбце days_since_prev       найдено  290461  дубликатов
В столбце event_id              найдено  268184  дубликатов
В столбце event_name            найдено  275363  дубликатов
В столбце event_type_main       найдено  290604  дубликатов
В столбце service_name          найдено  290575  дубликатов
В столбце region_name           найдено  290530  дубликатов
В столбце city_name             найдено  290259  дубликатов
В столбце revenue_rub           найдено  247522  дубликатов


### 2.3 *Промежуточный вывод*

## 3. Создание профиля пользователя <a id=3></a>

### 3.1 *___________*

### 3.2 *___________*

## 4. Предобработка данных <a id=4></a>

### 4.1 *Исследование признаков первого заказа и их связи с возвращением на платформу*

#### 4.1.1 _________

#### 4.1.2 _________

#### 4.1.3 _________

### 4.2 *Исследование поведения пользователей через показатели выручки и состава заказа*

#### 4.2.1 _________

#### 4.2.2 _________

#### 4.2.3 _________

### 4.3 *Исследование временных характеристик первого заказа и их влияния на повторные покупки*

#### 4.3.1 _________

#### 4.3.2 _________

### 4.4 *Корреляционный анализ количества покупок и признаков пользователя*

## 5. Общие выводы и рекомендации <a id=5></a>